# House Prices: Advanced Regression Techniques
### Home Price Prediction (Kaggle Competition)
**Author:** Eduard Trubichkin  
**Date:** march 2026  
**Description:** Complete data analysis, preprocessing, modeling (sklearn, XGBoost), and linear regression implementation from scratch in numpy.

In [ ]:
import pandas as pd
import numpy as np

Creating 2 objects of the `pandas.DataFrame` class: train and test. These objects can be represented as a table in Excel, with columns and rows.

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

We use the method of the instance object `info()` (this method is available to all instances of the DataFrame class). We get information about the file, which states `RangeIndex: 1460 entries, 0 to 1459`, which means that there are only 1460 rows in the table. We also see `Data columns (total 81 columns)`, which means that there are 81 columns in the file. The names of the columns from 0 to 80 inclusive are listed below. `Non-Null Count` means the number of non-empty values. For example, `LostFrontage` has only 1201 missing positions out of 1460. `Dtype` means the data type.

In [ ]:
train.info()

Let's look at the basic statistics of SalePrice prices. This is done using the `describe()` method, which is a method of the `DataFrame` instance. This method is applicable both to the `DataFrame` object and to a specific column. Since we are specifically interested in price statistics, we will apply the method to a specific column.   
`pandas.DataFrame` is designed so that its columns can be accessed by name in two ways:   
1) `train['SalePrice']` - always works   
2) `train.SalePrice` - does not work if there are spaces or special characters in the name.   

The basic statistics include parameters such as: 
- count (the total number of parameters to make sure there are no gaps in the column)
- mean (average price)
- std (standard deviation, a measure of how much the values deviate from the average)
- min/max (extreme values)
- 25% (meaning that 25% of houses are cheaper, 75% more expensive)
- 50% (half of the houses are cheaper, half are more expensive)
- 75% (75% of the houses are cheaper, 25% are more expensive)

In [ ]:
train['SalePrice'].describe()

The results should pay due attention to the variation from 75% to max. If the spread is large, this is a clear sign of outliers. That is, there are several extremely expensive houses in the data. The model may learn incorrectly from such outliers, so they are usually removed from the sample or transformations (logarithm) are used to smooth out the impact.  

Based on the data obtained, we can conclude that there is a huge gap between 75% and the maximum in the dataset (this needs to be fixed), and there are also many gaps in a number of parameters (this also needs to be fixed). Let's start working with the passes. Let's calculate their number (in absolute and percentage terms) and decide what to do with them (either deleting a column if there are too many gaps, or deleting a row if there are few gaps, or we can fill in something).

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

`isnull()` is a method of the `DataFrame` instance. It returns a new object of the same size, but consisting of Boolean values: `True` where there is no value, `False` where there is a value. The `sum()` applied to the `DataFrame` instance counts all `True` columns and outputs the sum: how many gaps are in each column. We end up with a `Series` object. The `Series` instance is just one component of the `DataFrames` instance, one column. Visually it looks like two columns, but in fact the first column is an index, which can be anything, even a number, even a row.   
After the table has been converted to a table containing values indicating the number of gaps, it is necessary to filter the column. To do this, the `missing=missing[missing>0]` construction is used, which leaves only those records where there is at least one pass. After that, the `sort_values()` method is applied (which applies to both `Series` and `DataFrame` instances), which sorts the remaining non-zero values. The argument `ascending=True` (used by default) is ascending sorting, `ascending=False` is descending sorting.   

After getting the absolute value of the number of passes, it is necessary to get the value in relative form. To do this, you can use the following algorithm: divide the instance of `Series` by the length of `train` and multiply by 100. `len(train)` returns the number of rows in the instance of `DataFrame'. Vectorized operations are available in pandas, that is, when dividing an instance of a Series, the division occurs piecemeal.

In [ ]:
missing_percent = train.isnull().sum() / len(train) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)
print(missing_percent)

Columns that have >90% omissions can be completely deleted, as they are almost completely empty and hardly carry any valuable information. However, before deleting it, you need to carefully look at the meaning of the attribute. Sometimes there are signs that have a null value, but not because they are empty, but because they indicate the absence/negation of some parameter. For example, the "Alley" parameter indicates the presence or absence of an "alley". And in this particular case, it would be more rational to replace the pass with "NoAlley".   

In general, in "train.csv", the "Alley" column is mostly set to "NA", which is not a skip. But when pandas reads CSV using pandas.read_csv(), it has a list of values that are considered skipped by default. This list includes, for example: 'NA', 'null', 'NaN', 'None', an empty string, and some others. That is why, when the `isnull()` method was applied, all these cells acquired the value `True`. Therefore, during data processing, it is necessary to take into account such moments and carefully monitor which columns really need to be removed, and in which columns the values need to be subdivided.   

Let's take a closer look at the "Alley" column. In order to understand the column structure, you can use the `value_counts()` method with the `dropna` argument. This is the `Series` instance method, it counts how many times each unique value occurs in a column. The result is also a `Series` object, where the indexes are unique values and the values are their frequencies. The `dropna` argument determines whether to include skips in the count. The default value is `True`. If it is necessary to include gaps in the calculation, then `dropna=False` is used.

In [ ]:
print(train['Alley'].value_counts(dropna=False))

The results show that the "Alley" column, in addition to the "Nan" values, has values such as "Grvl" (gravel) and "Pave" (asphalt). Thus, it can be understood that the "NA" in the "Alley" column does not mean emptiness, but rather the absence of an alley near the house. This means that it is necessary to replace the values of "NA" with "NoAllay".   

To change the values, you can use the `fillna()` method, which can be applied to both `DataFrame` and `Series` objects. The purpose of this method is to fill in the missing values in the data. A string is used as an argument, which will be used to replace all the omissions.

In [ ]:
train['Alley'] = train['Alley'].fillna('NoAlley')
print(train['Alley'].value_counts(dropna=False))

As you can see from the result, the string "NoAlley" is now used instead of an empty value.   

So, as mentioned earlier, columns with >90% omissions should be deleted if the omissions do not carry information. If the column has an average or small number of gaps, then they also need to be filled in, but using more "accurate" methods. For example, the median of the group, zero, mode, and so on, so as not to distort the distribution.   
In any case, there should be no gaps at all in the training sample, since most machine learning models cannot work with missing values. If there is at least one "NaN" left in the data, an error will be received when trying to train the model. Therefore, it is necessary to continue to identify gaps that carry information.  

There are a total of 19 columns in the CSV with at least one pass. The "Alley" column has already been disassembled, so we move on to the next columns.   
The file will help with the analysis "data_description.txt ", which contains a description of all the signs. If you open it and find, for example, "Alley" there, you can see that NA (or skip) means "No alley access", and not lack of data. Therefore, the analysis of the remaining columns will be based on a file with a description of the features.

- **PoolQC**: 99.520548% passes; "NA" means that there is no pool; Replace with "NoPool"
- **MiscFeature**: 96.301370% passes; "NA" means that there are no additional buildings; Replace with "NoMiscFeature"
- **Fence**: 80.753425% passes; "NA" means that there is no fence; Replace with "NoFence"
- **MasVnrType**: 59.726027% passes; "None" means that there is no cladding; Replace with "NoMasVnrType"
- **FireplaceQu**: 47.260274% passes; "NA" means that there is no fireplace; Replaced by "NoFireplaceQu"
- **LotFrontage**: 17.739726% of passes; "LotFrontage" displays the length of the street along the site; The passes can be filled in with the median for the area, since in one area the plots are usually similar. 
- **GarageType**: 5.547945% passes; "NA" means there is no garage; let's replace it with "NoGarage".
- **GarageYrBlt**: 5.547945% passes; The year when the garage was built; a pass means that there is no garage; let's replace it with zero. 
- **GarageFinish**: 5.547945% passes; A sign indicating the finishing of the garage; "NA" means that there is no finishing; replace with "NoGarage".
- **GarageQual**: 5.547945% passes; The attribute indicates the quality of the garage; "NA" means that there is no garage; replace with "NoGarage". 
- **GarageCond**: 5.547945% passes; The sign indicates the condition of the garage; "NA" means that there is no garage; Replace with "NoGarage". 
- **BsmtExposure**: 2.602740% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement". 
- **BsmtFinType2**: 2.602740% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **BsmtQual**: 2.534247% of passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **BsmtCond**: 2.534247% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **BsmtFinType1**: 2.534247% passes; In this feature, "NA" means that there is no basement; Let's replace it with "NoBasement".
- **MasVnrArea**: 0.547945% of passes; This parameter describes the area of masonry cladding in square feet. Using the string `print(train['MasVnrArea'].value_counts(dropna=False) / len(train) * 100)` you can find out what values are in the column at all and what percentage of the total they occupy. The value "0.0" has 58.972603% of the total number of all values. While the value of "NaN" is 0.547945%. Based on the meaning of the parameter, it would be most logical to fill in the gaps with zeros, since if we fill them in, for example, with the median, the model will think that the house has cladding, although it actually will not be. Fill in all the gaps with the value zero.
- **Electrical**: 0.068493% of passes; This parameter has only 1 pass; therefore, you can simply delete the row with this pass, as this does not affect the overall picture.

In [ ]:
train['PoolQC'] = train['PoolQC'].fillna('NoPool')
print(train['PoolQC'].value_counts(dropna=False))

In [ ]:
train['MiscFeature'] = train['MiscFeature'].fillna('NoMiscFeature')
print(train['MiscFeature'].value_counts(dropna=False))

In [ ]:
train['Fence'] = train['Fence'].fillna('NoFence')
print(train['Fence'].value_counts(dropna=False))

In [ ]:
train['MasVnrType'] = train['MasVnrType'].fillna('NoMasVnrType')
print(train['MasVnrType'].value_counts(dropna=False))

In [ ]:
train['FireplaceQu'] = train['FireplaceQu'].fillna('NoFireplace')
print(train['FireplaceQu'].value_counts(dropna=False))

In the context of LotFrontage, all gaps will be filled in with the median for the area. In one area, the sections are usually similar in size, so the length of the frontal part should be close.   

The implementation consists of several stages:   
1) Calculating the median "LotFrontage" for each district  
2) Filling in gaps using `transform`  
3) Handling a special case in which all values of the "LotFrontage" column may be skipped in some area 
4) Verification 

The first and second steps can be combined:

In [ ]:
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

To implement filling in gaps with the median, the `groupby()` method is used, which groups all DataFrame rows by unique values. In this case, the value "Neighborhood" is passed, and it is by the unique values of this column that the grouping will take place. The result of `groupby()` is a special `DataFrameGroupBy` object that stores information about groups. In fact, this is the same object, but sorted so that first there are lines with one "Neighborhood" value, then with another, then with a third, and so on.   

After grouping, you need to select the specific column you want to work with. In this case, it is the "LotFrontage" column. The entry `['LotFrontage']` means that you need to take only the value of the column "LotFrontage" from each group. After that, a `SeriesGroupBy` object is obtained - a set of series (one for each group), each of which contains the "LotFrontage" values for houses in a particular area. That is, the remaining 79 columns out of 81 are cut off. And the work is carried out with only 2 columns - "Neighborhood" and "LotFrontage". This results in a single `SeriesGroupBy` object that contains tables whose indexes are a unique ID value, and the values are "LotFrontage" values. But the groups are formed according to the unique meaning of the "Neighborhood" attribute.  

Next, the `transform()` method is used, which applies the function to each group, and then collects everything in a DataFrame. In `transform()`, `lambda` is passed, a small function without a name. It looks like: `lambda arguments: expression`. And in this case, the function has the form `lambda x: x.fillna(x.median())`, where `x` is the group `SeriesGroupBy`. And for each such group, the `fillna()` method is applied with the `median()` argument, which calculates the median. And so the gaps in each group are filled in with the median value of each group.   

You also need to prepare a backup plan. It may happen that in the "LotFrontage" there will be no data at all in one or more "Neighborhoods". And the median is calculated using only known values, zeros/omissions are ignored. In this case, it would be logical to replace the gaps with a common median.  

In [ ]:
if train['LotFrontage'].isnull().any():
    global_median = train['LotFrontage'].median()
    train['LotFrontage'].fillna(global_median, inplace=True)

print(train['LotFrontage'].isnull().sum())

Let's create an algorithm that checks if there is at least one pass in the "LotFrontage". This is implemented using the `any()` method. It returns `True` if there is at least one `True` in the iterated object. If the condition is true, then calculate the median for the entire "LotFrontage" column. After that, we fill in the remaining gaps with this value. The `inplace=True` argument means that the object is changed immediately, without a copy.   

Next, we will replace the omissions on the garage signs: the omissions in the year of construction column by 0, and in the rest - "NoGarage".

In [ ]:
train['GarageYrBlt'] = train['GarageYrBlt'].fillna(0)
print(train['GarageYrBlt'].value_counts(dropna=False))

In [ ]:
garage_str_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for x in garage_str_cols:
    train[x] = train[x].fillna('NoGarage')

for x in garage_str_cols:
    print(train[x].isnull().sum())

Now we'll do the same with the basements.

In [ ]:
bsmt_str_cols = ['BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1']
for x in bsmt_str_cols:
    train[x] = train[x].fillna('NoBasement')

for x in bsmt_str_cols:
    print(train[x].value_counts(dropna=False))

Now let's solve the last 2 problems: "MasVnrArea" and "Electrical". In "MasVnrArea" all the omissions will be replaced with zeros, but in the case of "Electrical" it is necessary to delete the row from the data in which there is a pass.

In [ ]:
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
print(train['MasVnrArea'].isnull().sum())

In [ ]:
train.dropna(subset='Electrical', inplace=True)
print(train['Electrical'].isnull().sum())

To delete a line where there is a gap, you can use the `dropna()` method. As an argument, you need to specify the column you want to target. And all rows where there are gaps in the specified column will be deleted.   
Make sure that there are no more gaps left in the training file.

In [ ]:
print(train.isnull().sum().sum())

Now that all the gaps in the data are filled in, you can start logarithming the outliers. As previously noted, there is a wide variation between 75% and max in the coaching data. Therefore, it is necessary to apply the formula RMSLE - Root Mean Squared Logarithmic Error.   

In regression tasks, you need to evaluate how well the model predicts the target value. Consider three related metrics.  

**MSE - Mean Squared Error**
The most basic loss function that is used in model training. It penalizes the model for large errors especially strongly (due to squaring).
$$ \mathrm{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 $$

where:    
$y_i$ - true value    
$\hat{y}_i$ - predicted value    
${n}$ - number of objects   

The MSE units are the square of the target variable (for example, dollars²). This is inconvenient to interpret.   

**RMSE - Root Mean Squared Error**
To return to the original units of measurement (dollars or another currency), extract the square root of the MSE:
$$ \mathrm{RMSE} = \sqrt{\mathrm{MSE}} = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 } $$

RMSE shows how wrong the model is on average in absolute terms. For example, if RMSE = 50,000, then the model is wrong by about $50,000 in one direction or another.   
However, RMSE has a problem: it is sensitive to emissions. One very expensive house, which we predicted badly, can greatly spoil the metric, even if the error on the remaining 99% of houses is small.   

**RMSLE - Root Mean Squared Logarithmic Error**
The RMSLE metric is calculated not from the prices themselves, but from their logarithms.
$$ \mathrm{RMSLE} = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} \left( \ln(y_i + 1) - \ln(\hat{y}_i + 1) \right)^2 } $$
There are several reasons why you need to use a metric that uses logarithms.:   
1) RMSLE penalizes for percentage error, not for absolute error. An error of 100,000 for a house priced at 1,000,000 (10%) will be approximately equal to an error of 10,000 for a house priced at 100,000 (also 10%). This is fairer when prices are highly dispersed.
2) There is a big gap between 75% and the maximum value in this dataset. If you teach the model to predict the usual price, these several houses will "drag" the model on themselves. Logarithm compresses the long tail of the distribution, making it more symmetrical. The model stops "panicking" about rare expensive houses and captures general patterns better.

That is, logarithmization allows you to make large numbers relatively closer to small ones, because the growth rate of the logarithm slows down. Logarithmization "compresses" the distribution and makes it more symmetrical - it turns multiplicative differences (in times) into additive ones, which is convenient for many models.  

In [ ]:
train['SalePrice_log'] = np.log1p(train['SalePrice'])
print(train['SalePrice_log'].describe())

Now we have a logarithmic goal. You can proceed to encoding categorical features.   
Machine learning models do not understand text. They need numbers. Therefore, each category needs to be converted into a numerical representation. There are two main approaches: 
- One-Hot Encoding - for nominal signs (where there is no order). In our case, such a feature is "Neighborhood". It makes no sense to say that district "A" is bigger than district "B".  
- Ordinal Encoding - for ordinal features where categories have a natural order. For example, finishing quality (Exterior): Ex (Excellent) > Gd (Good) > TA (Average) > Fa (Fair) > Po (Poor). Here you can replace the values with numbers, preserving the order.

Based on the file data_description.txt The following **ordinal** signs can be distinguished: OverallQual, OverallCond, ExterQual, ExterCond, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1 / BsmtFinType2, HeatingQC, KitchenQual, FireplaceQu, GarageFinish, GarageQual / GarageCond, PoolQC, Functional, LandSlope, PavedDrive, LotShape, Fence.   

Let's use the `unique()` method of the `Series` instance to view the list of unique values in all **ordinal** columns and see if everything is in order and if the data is ready for encoding.   
Also, for convenience, `tolist()` will be used to convert the NumPy array (which returns `unique()`) to a regular Python list.

In [ ]:
ordinal = ['OverallQual', 'OverallCond', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Functional', 'LandSlope', 'PavedDrive', 'LotShape', 'Fence']
for x in ordinal:
    print(f"{x}: {train[x].unique().tolist()}")

Now you can proceed to encoding the **ordinal** features. For this, a dictionary will be used, the key of which is a feature, and the value is an array of all unique values from worst to best.

In [ ]:
ordinal_mapping = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NoBasement', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NoGarage', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['NoPool', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'PavedDrive': ['N', 'P', 'Y'],
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'Fence': ['NoFence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
}

Now let's define the list of columns on which the model will be trained. Let's start with the numeric columns. To do this, use the `select_dtypes()` method of the `DataFrame` instance. This method accepts a list of the required types `include=['int64', 'float64']`. Next, we apply the `columns` attribute, which returns an `Index` object with the names of all columns. And then we'll use the `tolist()` method to convert this object into a regular Python list.   

We get a list that contains all the names of the numeric columns. But 3 columns are superfluous - Id (so it has no value), SalePrice (since it is a target variable that should not be in the training sample) and SalePrice_log (also a target variable, but in a logarithmic representation).    
In order to remove them, we will use the **list inclusion**. We take an array with the names of all numeric columns and apply a construction that iterates through the array, and if the object being iterated over is not equal to the object in the array of these 3 features, then add it to the new array.

In [ ]:
numeric_features = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log']]
print(numeric_features)

Now let's create an array of ordinary attributes. All these signs have already been listed earlier.

In [ ]:
ordinal_features = list(ordinal_mapping.keys())
print(ordinal_features)

And the final stage will be the formation of an array of nominal features.

In [ ]:
categorical_features = train.select_dtypes(include=['object']).columns.tolist()
categorical_features = [col for col in categorical_features if col not in ordinal_features]
print(categorical_features)

In the end, you should get 3 Python arrays: numeric parameters, ordinal parameters and categorical parameters.